In [ ]:
!pip install umap-learn

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import plotly.express
import seaborn
import sklearn
import sklearn.preprocessing
import sklearn.decomposition
import umap

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s wine-quality
!ls wine-quality/

## Data

Load `wine-quality/wineQualityReds.csv` in a Dataframe `df` and display the 10 first examples.

In [ ]:
# Your code here

### Solution

In [ ]:
df = pandas.read_csv("wine-quality/wineQualityReds.csv").iloc[:, 1:]
df.head(10)
df.shape

## Feature Extraction

In order to use sklearn, put, without the quality (considered as the target), the features in a numpy array `X`.

In [ ]:
# Your code here

### Solution

In [ ]:
X = df.drop(columns="quality").values
print("Data shape:", X.shape)

## Normalization

Use [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to normalize your data so that, for each feature, the mean is 0 and the standard deviation is 1.

In [ ]:
# Your code here

### Solution

In [ ]:
scaler = sklearn.preprocessing.StandardScaler()
X_scaled = scaler.fit_transform(X)

## PCA

- Perform a PCA, keeping only the first 2 components which you will store in `X_pca`. Create an instance of scikit-learn's [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html?highlight=pca#sklearn.decomposition.PCA) class and use the [`fit_transform`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html?highlight=pca#sklearn.decomposition.PCA.fit_transform) method.

- Display the amount of variance retained by each component. For this you can use the `explained_variance_ratio_` attribute which is calculated after an `fit` or `fit_transform`.

In [ ]:
# Your code here

### Solution

In [ ]:
pca = sklearn.decomposition.PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
# pca.fit(X_scaled)
# X_pca = pca.transform(X_scaled)

print("Explained Variance:",
      ", ".join(f"{v:.2f}" for v in pca.explained_variance_ratio_))
print("Cumulated Explained Variance:",
      ", ".join(f"{v:.2f}"
      for v
      in numpy.cumsum(pca.explained_variance_ratio_)))

components_df = pandas.DataFrame(pca.components_, columns=df.columns[:-1])
components_df

## Plotting

Display `X_pca` with `matplotlib` and `seaborn`

In [ ]:
# Your code here

### Solution

In [ ]:
print("With seaborn")
seaborn.jointplot(x=X_pca[:,0] , y=X_pca[:,1])
plt.show()

print("With matplotlib")
plt.plot(X_pca[:, 0], X_pca[:, 1], ".")
plt.show()

print("With plotly")
plotly.express.scatter(X_pca, x=0, y=1, width=600)

## Outliers
A few points seem to stand out from the crowd.

Identify their index in order to display them from `df`.

In [ ]:
# Your code here

### Solution

In [ ]:
# The one at max value on the first PCA component.
max_pc0 = numpy.argmax(X_pca[:, 0])
print(max_pc0)
print(df.iloc[max_pc0, :])

In [ ]:
# The 5 top value for second PCA component.
top5_pc1 = numpy.argsort(-X_pca[:, 1])[:5]
df.iloc[top5_pc1, :]

## The Good, the Bad and the Ugly
Start by visualising the distribution of the qualities of the wines (via a barplot for example).

Then use the quality information to separate your wines into three groups:
 * Bad wines (quality < 5)
 * Average wines (quality 5 and 6)
 * Good wines (quality >= 7)

Then display your dimension reduction via a scatter plot, indicating with the colour of the point its quality.

In [ ]:
# Your code here

### Solution

In [ ]:
# Quality distribution
seaborn.countplot(x=df['quality'])
plt.show()

In [ ]:
bad_wine_ind  = df['quality'] < 5
mean_wine_ind = (df['quality'] >= 5) & (df['quality'] < 7)
good_wine_ind = df['quality'] >= 7

def display_projection(data):
  f, ax = plt.subplots(figsize=(10, 10))
  plt.scatter(data[mean_wine_ind, 0], data[mean_wine_ind, 1], color = "b",
              alpha=0.05)
  plt.scatter(data[bad_wine_ind, 0], data[bad_wine_ind, 1], color = "r")
  plt.scatter(data[good_wine_ind, 0], data[good_wine_ind, 1], color ="g")
  plt.show()

display_projection(X_pca)

## Non-linear projection

Use UMAP model of the umap package (which is used as PCA) to calculate a non-linear projection. Compare it to the projection obtained by PCA.

In [ ]:
# Your code here

### Solution

In [ ]:
display_projection(umap.UMAP().fit_transform(X_scaled))

In [ ]:
styling = dict(template="seaborn", width=800, height=600)

def show(fig):
  fig.update_traces(marker=dict(line=dict(width=0.1)),
                    selector=dict(mode='markers'))
  fig.show()


X_umap = umap.UMAP(n_components=3).fit_transform(X_scaled)
fig = plotly.express.scatter_3d(pandas.DataFrame(X_umap),
                                x=0,
                                y=1,
                                z=2,
                                size_max=0.1,
                                color=df["quality"],
                                opacity= 0.6,
                                **styling)
fig.update_traces(marker=dict(size=2.5,
                              line=dict(width=1, color=df["quality"])),
                  selector=dict(mode='markers'))
show(fig)